# H₂ Dissociation Curve — VQE vs ADAPT-VQE vs Exact

We compute the ground state energy of H₂ across a range of bond lengths (0.3Å to 3.0Å)
using three methods: exact classical diagonalization, standard UCCSD-VQE, and ADAPT-VQE.

This is a canonical benchmark in quantum chemistry:
- Near equilibrium (~0.735Å) all methods agree well
- Near the dissociation limit, single-reference methods (UCCSD) fail
- The exact curve captures the proper dissociation behavior
- Chemical accuracy threshold: 1.6 mHa (0.0016 Ha)

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

from qiskit_algorithms import VQE, AdaptVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock
from qiskit.primitives import Estimator

print("All imports OK")

## Step 2 — Setup: Bond Lengths & Compute All Curves

We sweep 20 bond lengths from 0.3Å to 3.0Å.
At each point we run: exact diagonalization, UCCSD-VQE, and ADAPT-VQE.

In [ ]:
bond_lengths = np.linspace(0.3, 3.0, 20)
exact_energies = []
uccsd_energies = []
adapt_energies = []
uccsd_params = []
adapt_params = []

estimator = Estimator()

print(f"Running dissociation curve for {len(bond_lengths)} bond lengths...")
print(f"Range: {bond_lengths[0]:.2f}Å to {bond_lengths[-1]:.2f}Å")
print("-" * 60)

for i, dist in enumerate(bond_lengths):
    mol = f"H 0.0 0.0 0.0\nH {dist:.3f} 0.0 0.0\n"
    driver = PySCFDriver(atom=mol, basis="sto3g")
    problem = driver.run()

    num_spatial = problem.num_spatial_orbitals
    num_particles = problem.num_particles
    nuclear_repulsion = problem.nuclear_repulsion_energy()

    hamiltonian = problem.hamiltonian
    second_q_op = hamiltonian.second_q_op()
    mapper = JordanWignerMapper()
    qubit_op = mapper.map(second_q_op)

    exact_matrix = qubit_op.to_matrix()
    exact_eig = np.linalg.eigh(exact_matrix).eigenvalues[0]
    exact_energies.append(exact_eig + nuclear_repulsion)

    initial_state = HartreeFock(
        num_spatial_orbitals=num_spatial,
        num_particles=num_particles,
        qubit_mapper=mapper,
    )

    ansatz = UCCSD(
        num_spatial_orbitals=num_spatial,
        num_particles=num_particles,
        qubit_mapper=mapper,
        initial_state=initial_state,
        excitations=[1, 2],
    )

    vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=COBYLA(maxiter=300))
    result_uccsd = vqe.compute_minimum_eigenvalue(qubit_op)
    uccsd_energies.append(result_uccsd.eigenvalue.real + nuclear_repulsion)
    uccsd_params.append(ansatz.num_parameters)

    pool_ops, _ = ansatz.excitation_ops()
    adapt_vqe = AdaptVQE(
        estimator=estimator,
        ansatz=None,
        optimizer=COBYLA(maxiter=300),
        initial_operator_pool=pool_ops,
        threshold=1e-6,
    )
    result_adapt = adapt_vqe.compute_minimum_eigenvalue(qubit_op)
    adapt_energies.append(result_adapt.eigenvalue.real + nuclear_repulsion)
    adapt_params.append(len(result_adapt.eigenvalue_history))

    print(f"  [{i+1:2d}/{len(bond_lengths)}] d={dist:.3f}Å | exact={exact_energies[-1]:.6f} | "
          f"uccsd={uccsd_energies[-1]:.6f} | adapt={adapt_energies[-1]:.6f}")

print("-" * 60)
print("Done!")

## Step 3 — Dissociation Curve Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(bond_lengths, exact_energies, 'k-', linewidth=2.5,
        label='Exact (Classical)', zorder=5)
ax.plot(bond_lengths, uccsd_energies, 'g-s', markersize=5, linewidth=1.5,
        label='UCCSD-VQE', zorder=4)
ax.plot(bond_lengths, adapt_energies, 'b-o', markersize=5, linewidth=1.5,
        label='ADAPT-VQE', zorder=4)

eq_idx = np.argmin(np.abs(bond_lengths - 0.735))
ax.scatter([bond_lengths[eq_idx]], [exact_energies[eq_idx]],
           color='red', s=100, zorder=6, marker='*',
           label=f'Equilibrium ({bond_lengths[eq_idx]:.3f}Å)')

ax.axvline(x=0.735, color='red', linestyle=':', alpha=0.5, linewidth=1)

ax.set_xlabel('Bond Length (Angstrom)', fontsize=13)
ax.set_ylabel('Ground State Energy (Hartree)', fontsize=13)
ax.set_title('H₂ Dissociation Curve\nExact vs UCCSD-VQE vs ADAPT-VQE', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

textstr = (f'Chemical accuracy: 1.6 mHa\n'
            f'Exact at eq: {exact_energies[eq_idx]:.6f} Ha')
ax.text(0.97, 0.03, textstr, transform=ax.transAxes,
         fontsize=10, verticalalignment='bottom', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('vqe_h2/dissociation_curve.png', dpi=150)
plt.show()

print("Dissociation curve saved to dissociation_curve.png")

## Step 4 — Error Analysis: Energy Error vs Bond Length

In [ ]:
uccsd_errors = [abs(e - ex) for e, ex in zip(uccsd_energies, exact_energies)]
adapt_errors = [abs(e - ex) for e, ex in zip(adapt_energies, exact_energies)]

fig, ax = plt.subplots(figsize=(12, 6))

ax.semilogy(bond_lengths, uccsd_errors, 'g-s', markersize=5, linewidth=1.5,
            label='UCCSD-VQE error')
ax.semilogy(bond_lengths, adapt_errors, 'b-o', markersize=5, linewidth=1.5,
            label='ADAPT-VQE error')

chemical_accuracy = 0.0016
ax.axhline(y=chemical_accuracy, color='red', linestyle='--', linewidth=1.5,
           label=f'Chemical accuracy ({chemical_accuracy} Ha)')

ax.set_xlabel('Bond Length (Angstrom)', fontsize=13)
ax.set_ylabel('|Error| vs Exact (Hartree, log scale)', fontsize=13)
ax.set_title('VQE Error vs Bond Length: UCCSD vs ADAPT', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-8, 1e0)

plt.tight_layout()
plt.savefig('vqe_h2/dissociation_error.png', dpi=150)
plt.show()

print(f"Max UCCSD error: {max(uccsd_errors):.6f} Ha at d={bond_lengths[np.argmax(uccsd_errors)]:.3f}Å")
print(f"Max ADAPT error: {max(adapt_errors):.6f} Ha at d={bond_lengths[np.argmax(adapt_errors)]:.3f}Å")

## Step 5 — Parameter Count (Circuit Sparsity) Across Dissociation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(bond_lengths, uccsd_params, 'g-s', markersize=5, linewidth=1.5,
       label='UCCSD-VQE parameters')
ax.plot(bond_lengths, adapt_params, 'b-o', markersize=5, linewidth=1.5,
       label='ADAPT-VQE parameters')

ax.set_xlabel('Bond Length (Angstrom)', fontsize=13)
ax.set_ylabel('Number of Parameters', fontsize=13)
ax.set_title('Ansatz Complexity Across Dissociation: UCCSD vs ADAPT', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vqe_h2/dissociation_params.png', dpi=150)
plt.show()

print(f"Avg UCCSD params: {np.mean(uccsd_params):.1f}")
print(f"Avg ADAPT params: {np.mean(adapt_params):.1f}")
print(f"Avg sparsity gain: {100*(1-np.mean(adapt_params)/np.mean(uccsd_params)):.1f}%")

## Step 6 — Summary Table

In [ ]:
print("=" * 70)
print(f"{'DISSOCIATION CURVE SUMMARY':^70}")
print("=" * 70)
print(f"Bond length range: {bond_lengths[0]:.2f}Å to {bond_lengths[-1]:.2f}Å ({len(bond_lengths)} points)")
print(f"Equilibrium bond length: {bond_lengths[eq_idx]:.3f}Å")
print(f"Equilibrium energy (exact): {exact_energies[eq_idx]:.12f} Ha")
print("-" * 70)
print(f"{'Metric':<35} {'UCCSD-VQE':<18} {'ADAPT-VQE':<18}")
print("-" * 70)
print(f"{'Max error across all points (Ha)':<35} {max(uccsd_errors):<18.8f} {max(adapt_errors):<18.8f}")
print(f"{'Mean error across all points (Ha)':<35} {np.mean(uccsd_errors):<18.8f} {np.mean(adapt_errors):<18.8f}")
print(f"{'Avg number of parameters':<35} {np.mean(uccsd_params):<18.1f} {np.mean(adapt_params):<18.1f}")
print(f"{'Points within chemical accuracy':<35} {sum(e < 0.0016 for e in uccsd_errors):<18} {sum(e < 0.0016 for e in adapt_errors):<18}")
print("=" * 70)